# Corponi Replication: Phase 2 (Supervised Fine-tuning on WESAD)

## Overview
This notebook implements the fine-tuning loop for WESAD with a full metrics report.

### Goal
Train the final classifier and export complete results for thesis comparison.

In [ ]:
# 1. Environment Setup
import os, sys, shutil, glob, pickle, numpy as np
from IPython.display import FileLink

WORK_DIR = '/kaggle/working/e4selflearning'
if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)
print("Cloning repository...")
!git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

print("Installing dependencies...")
!pip install -q "numpy==1.26.4" "scipy==1.12.0" "pandas<2.2" biosppy flirt hrv-analysis neurokit2 peakutils lightning torchmetrics

if WORK_DIR not in sys.path: sys.path.insert(0, f"{WORK_DIR}/src")
os.chdir(WORK_DIR)

### 2. Generate Custom Training Script
We create `train_ann_fixed.py` with the complete metrics report.

In [ ]:
script_content = r'''
import argparse, os, pickle, numpy as np, torch, re
from torch.utils.data import DataLoader
from tqdm import tqdm
from timebase import criterions
from timebase.models.models import get_models
from timebase.data.dataset import ClassificationDataset
from timebase.utils import utils
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

class WESAD_ClassificationDataset(ClassificationDataset):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.positive_classes = [1.0]
        
    def session_id(self, filename: str) -> int:
        match = re.search(r'[sS](\d+)', filename)
        return int(match.group(1)) if match else 0

def load_pre_trained(args, classifier):
    target_file = None
    for r, d, f in os.walk(args.path2pretraining_res):
        for candidate in ["model_best.pt", "model_state.pt"]:
            if candidate in f:
                target_file = os.path.join(r, candidate)
                break
        if target_file: break
    if not target_file: raise FileNotFoundError("ERROR: Weights not found!")
    ckpt = torch.load(target_file, map_location=args.device)
    sd = classifier.sslearner.state_dict()
    sd.update({k: v for k, v in ckpt["model"].items() if any(m in k for m in ["channel_embedding", "feature_encoder"])})
    classifier.sslearner.load_state_dict(sd)
    print(f" [OK] Weights loaded: {target_file}")

def main(args):
    utils.set_random_seed(args.seed)
    args.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(args.output_dir, exist_ok=True)
    
    args.verbose = 1; args.use_wandb = False; args.save_plots = False
    args.critic_score_lambda = 0.0; args.reuse_stats = False; args.clear_output_dir = False
    
    cli_dataset_path = args.dataset
    args.task_mode = 2
    utils.load_args(args, dir=args.path2pretraining_res)
    args.dataset = cli_dataset_path
    args.task_mode = 1
    args.e4selflearning = True
    
    with open(os.path.join(args.dataset, "metadata.pkl"), "rb") as f:
        meta = pickle.load(f)
    args.ds_info = meta["ds_info"]
    args.input_shapes = {c: [512 * args.ds_info["channel_freq"][c]] for c in ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']}
    
    stats = None
    stats_path = os.path.join(args.dataset, "stats.pkl")
    if os.path.exists(stats_path):
        with open(stats_path, "rb") as f:
            sl = pickle.load(f)
            stats = sl[list(sl.keys())[0]]
    if stats is None:
        stats = {c: {'min': -10, 'max': 10, 'mean': 0, 'std': 1, 'median': 0, 'iqr': 1} for c in ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']}

    corrected_paths = []
    for p in meta["sessions_paths"]:
        parts = p.replace("\\", "/").split("/")
        corrected_paths.append(os.path.join(args.dataset, parts[-2], parts[-1]))
    all_paths = np.array(corrected_paths)
    labels_dict = {k: np.array(v) for k, v in meta["sessions_labels"].items()}
    
    train_subs = ["S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10", "S11"]
    test_subs  = ["S13", "S14", "S15", "S16", "S17"]
    args.num_train_subjects = len(train_subs)
    #
    args.num_train_subjects = len(train_subs)
    
    mask_train = np.array([any(s in p.upper() for s in train_subs) for p in all_paths])
    mask_test = np.array([any(s in p.upper() for s in test_subs) for p in all_paths])
    
    ds_train = WESAD_ClassificationDataset(args=args, filenames=all_paths[mask_train], labels={k: v[mask_train] for k, v in labels_dict.items()}, rec_ids=all_paths[mask_train], stats=stats, recording_id_str_to_num={p: i for i, p in enumerate(np.unique(all_paths[mask_train]))})
    ds_test = WESAD_ClassificationDataset(args=args, filenames=all_paths[mask_test], labels={k: v[mask_test] for k, v in labels_dict.items()}, rec_ids=all_paths[mask_test], stats=stats, recording_id_str_to_num={p: i for i, p in enumerate(np.unique(all_paths[mask_test]))})
    
    loader_train = DataLoader(ds_train, batch_size=args.batch_size, shuffle=True)
    loader_test = DataLoader(ds_test, batch_size=args.batch_size, shuffle=False)
    
    classifier, _ = get_models(args, summary=None)
    load_pre_trained(args, classifier)
    classifier.to(args.device)
    opt = torch.optim.AdamW(classifier.parameters(), lr=args.lr)
    crit = criterions.get_criterion(args)
    
    print(f" Training: {train_subs}\n Testing: {test_subs}")
    
    for epoch in range(1, args.epochs + 1):
        classifier.train()
        losses = []
        for batch in tqdm(loader_train, desc=f"Epoch {epoch}", leave=False):
            inputs = {k: v.float().to(args.device) for k, v in batch["data"].items()}
            targets = batch["target"].to(args.device)
            out, _ = classifier(inputs)
            loss = crit.criterion_classifier(y_true=targets, y_pred=out)
            loss.backward(); opt.step(); opt.zero_grad()
            losses.append(loss.item())
        if epoch % 10 == 0 or epoch == 1: print(f" Epoch {epoch}: Loss = {np.mean(losses):.4f}")

    classifier.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in tqdm(loader_test, desc="Evaluating"):
            inputs = {k: v.float().to(args.device) for k, v in batch["data"].items()}
            out, _ = classifier(inputs)
            y_true.extend(batch["target"].cpu().numpy())
            y_pred.extend((torch.sigmoid(out) >= 0.5).long().cpu().numpy())
    
    # --- FINAL REPORT ---
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    f1_m = f1_score(y_true, y_pred, average='macro')
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    
    print("\n" + "="*40)
    print(" CLASSIFICATION REPORT")
    print("="*40)
    print(f" Accuracy  : {acc*100:>6.2f}%")
    print(f" F1-Score  : {f1*100:>6.2f}%")
    print(f" Precision : {prec*100:>6.2f}%")
    print(f" Recall    : {rec*100:>6.2f}%")
    print("-"*40)
    print(" CONFUSION MATRIX:")
    print(cm)
    print("="*40)
    
    torch.save(classifier.state_dict(), os.path.join(args.output_dir, "finetuned_model.pt"))

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", type=str, required=True)
    parser.add_argument("--path2pretraining_res", type=str, required=True)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=0.0001)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--seed", type=int, default=42)
    main(parser.parse_args())
'''

with open("train_ann_fixed.py", "w") as f: f.write(script_content)


### 3. Data Ingestion & Execution

In [ ]:
import os, glob, shutil

ckpt_dirs = glob.glob('/kaggle/input/**/ckpt_sslearner', recursive=True)
PRE_RES = os.path.dirname(ckpt_dirs[0]) if ckpt_dirs else "/kaggle/working/e4selflearning/output_pretrain"
wesad_meta = glob.glob('/kaggle/input/**/wesad_segmented/metadata.pkl', recursive=True)
if not wesad_meta: raise FileNotFoundError("WESAD metadata not found!")
DATA_DIR = os.path.dirname(wesad_meta[0])

print(f"Pre-train: {PRE_RES}\nDataset: {DATA_DIR}\n\n--- TRAINING ---")
!PYTHONPATH=src python train_ann_fixed.py \
    --dataset {DATA_DIR} \
    --path2pretraining_res {PRE_RES} \
    --epochs 50 \
    --output_dir output_wesad_final

### 4. Export Results

In [ ]:
import shutil
from IPython.display import FileLink
if os.path.exists("output_wesad_final"):
    shutil.make_archive("wesad_final_results", 'gztar', "output_wesad_final")
    print("Archive created: wesad_final_results.tar.gz")
    display(FileLink("wesad_final_results.tar.gz"))